# Polar-only chip creation

This is the polar-focused counterpart to [chip_example.ipynb](chip_example.ipynb), using the same chip backend. Small north/south AOIs may cross ±82° or the antimeridian. Full-longitude pole caps are supported on polar-stereographic grids: the enclosing rectangle retains imagery and labels in its corners, without a circular mask. Pole integration tests and real-data review require HPC validation.

The user reports the updated chip tests passed on HPC, including seam regressions. Real-data seam Run All and overlay inspection remain pending. Edit source/label paths and AOIs together; latest-label discovery does not verify spatial association.

Dynamic output retains the original raster CRS and pixel lattice. Static-only output uses a 100 m, zero-anchored lattice in the AOI-center grid (LPS for polar centers). Acquisition tiles are intermediate, not the output grid. Select the project's GDAL-enabled kernel.


In [ ]:
from pathlib import Path
from datetime import datetime
import sys

repo_root = Path(str(Path.cwd().parent).replace(
    "/panfs/ccds02/nobackup", "/explore/nobackup"))
if not (repo_root / "lfm/data_processing").is_dir():
    raise FileNotFoundError("Run from the repository's top-level notebooks/ directory.")
sys.path.insert(0, str(repo_root))

from lfm.data_processing.chip import (
    AcquisitionGroupConfig, ChipConfig, GeographicAOI, LabelInput,
    NoSplitConfig, OutputModalityConfig, SourceSelector,
    chip_grid_family, default_chip_zoom, chip_request_from_aoi, create_chips,
)
from lfm.data_processing.tiling import (
    TileConfig, TileSourceConfig, VectorIndexBuildConfig,
    ensure_vector_index, resolve_notebook_source_index, discover_raster_paths,
)
from lfm.data_processing.tiling.product_ids import lunar_product_id_from_raster_path
from lfm.all_models.all_tasks.tiling_utils import make_static_source
from lfm.data_processing.chip.chip_notebook_utils import (
    latest_crater_label_path, read_source_grid, plot_chip_result,
)


## 1. Configure one study area

- `HEMISPHERE`: north/south example; `AOI_NWSE` uses IAU:30100 north, west, south, east. The north default encloses craters 13–20 in the supplied `(2).gpkg`; the south example requires its own labels.
- `RUN_SEAM_EXAMPLE`: add a second query to the same batch; enabled for north WAC by default. `SEAM_AOI_NWSE` extends the labeled polar area below 82°N using the same `(2).gpkg`. Disable it or change its bounds for other data.
- `RUN_POLE_EXAMPLE`: optional additional pole chip, disabled by default. `POLE_AOI_NWSE` must use west=-180, east=180 and reach ±90°. Labels must be complete throughout the enclosing rectangle, including corners outside the requested cap.
- `MODALITY`: `wac`, `nac`, or `static`. Static-only ignores the dynamic raster; otherwise `INCLUDE_STATIC` appends the canonical 63 bands.
- `SOURCE_RASTER`: original pre-tiling raster (TIFF or VRT). For the default WAC example, `None` discovers exactly one hemisphere-specific VRT in the tiling example's collection. NAC requires an explicit path.
- `WAC_LAYOUT`: `single` for the polar monochrome mosaic, or `vis_uv` for the usual seven-channel VIS/UV product. The selected VIS reference has five bands for `vis_uv`; its UV companion must be available.
- `DYNAMIC_GLOB`: optional companion-file pattern; by default index only the selected raster. For `vis_uv`, derive the product's VIS/UV TIFF pattern. Never index both a VRT and its backing TIFF as duplicate imagery.
- `LABEL_PATH`: explicit GPKG/TIFF, or `None` for the latest crater-notebook GPKG. Confirm the printed path; discovery does not match hemisphere/product. `LABEL_KIND`: `auto` (TIFF semantic), `semantic`, or `raster_instance`. `LABEL_LAYER` applies only to GPKG.
- `ZOOM_OVERRIDE`: acquisition override for both families; defaults are WAC/static LTM 5 / polar 4, NAC LTM 11 / polar 10. Seam queries split at ±82°. Final resolution follows the native source or static-only 100 m center-selected grid.
- `MAX_OUTPUT_PIXELS`: safety limit before indexing/acquisition; reduce the AOI for high-resolution NAC. `MAX_WORKERS`: chip workers, not index workers.


In [ ]:
PROJECT_DATA_DIR = Path("/explore/nobackup/projects/lfm")
HEMISPHERE = "north"  # "north" or "south".
MODALITY = "wac"  # "wac", "nac", or "static".
INCLUDE_STATIC = False  # Start with imagery only; verify static coverage separately.
SOURCE_RASTER = None  # Set an explicit Path for your source, especially NAC.
WAC_LAYOUT = "single"  # Polar mosaic is not the seven-band VIS/UV product.
DYNAMIC_GLOB = None  # Optional glob selecting this product's companion files.
POLAR_WAC_DIR = PROJECT_DATA_DIR / "processed_data/Lunar/Static_final/LROC/WAC/wac_glob_morf_mos"
STATIC_DATA_DIR = PROJECT_DATA_DIR / "staticLinks"
AOI_NWSE = (86.1, -48.7, 83.95, -19.5) if HEMISPHERE == "north" else (-85.9, -0.1, -86.1, 0.1)
RUN_SEAM_EXAMPLE = HEMISPHERE == "north" and MODALITY == "wac"  # Add the labeled north seam query.
SEAM_AOI_NWSE = (86.1, -48.7, 81.95, -19.5)  # Includes IDs 13–20 and crosses 82°N.
RUN_POLE_EXAMPLE = False  # Enable after confirming source and label coverage near the pole.
POLE_AOI_NWSE = (90, -180, 89.5, 180) if HEMISPHERE == 'north' else (-89.5, -180, -90, 180)
# Antimeridian alternative; confirm your imagery and labels cover this area:
# AOI_NWSE = (86.01, 179.98, 86.0, -179.98) if HEMISPHERE == 'north' else (-86.0, 179.98, -86.01, -179.98)
LABEL_PATH = PROJECT_DATA_DIR / "example_notebook_inputs" / "WAC_GLOBAL_P900N0000_100M.eqc.iau2.LPS_N_label_craters.gpkg"  # Set your HPC copy's path if different.
LABEL_KIND = "auto"  # For instance TIFFs use "raster_instance".
LABEL_LAYER = "craters"
SAMPLE_ID = f"{MODALITY}_{HEMISPHERE}_aoi_1"  # Unique per study area.
ZOOM_OVERRIDE = None
MAX_OUTPUT_PIXELS = 4_000_000  # Bounds per-band memory; static adds 63 bands.
MAX_WORKERS = 1
INDEX_WORKER_COUNT = 1
SPLIT_CONFIG = NoSplitConfig(seed=42)
OUTPUT_BASE_DIR = repo_root / "notebooks/outputs/chip_creation_polar"

### Optional full workflow

After validating the single example, replace the request list with multiple AOIs on the same source lattice and in the same hemisphere. Give each AOI a unique ID. Keep a shared product split-group key to prevent train/test leakage. Separate runs are needed for different source grids or acquisition zooms.


In [ ]:
# from lfm.data_processing.chip import MixedPercentageNumberSplitConfig, SplitCounts, SplitPercentages
# SPLIT_CONFIG = MixedPercentageNumberSplitConfig(
#     fixed_counts=SplitCounts(test=100), remaining_percentages=SplitPercentages(train=0.9, val=0.1),
#     seed=42,
# )
# MAX_WORKERS = 16
# Below, replace requests = [request] with requests built for your AOIs.


## 2. Resolve labels and inspect the output grid

No imagery acquisition occurs here. The realized rectangular raster may extend slightly beyond the input AOI due to reprojection and outward pixel rounding; labels must cover the entire realized grid. Review dimensions and label association before continuing. Canonical static coverage can be partial or absent; verified empty channels retain -32768 with warnings, while corrupt/missing inputs still fail.


In [ ]:
if HEMISPHERE not in ("north", "south") or MODALITY not in ("wac", "nac", "static"):
    raise ValueError("Choose a supported hemisphere and modality.")
if WAC_LAYOUT not in ("single", "vis_uv"):
    raise ValueError("WAC_LAYOUT must be 'single' or 'vis_uv'.")
requested_aoi = GeographicAOI(*AOI_NWSE)
family = chip_grid_family(requested_aoi)
expected_family = "lps_n" if HEMISPHERE == "north" else "lps_s"
if not (requested_aoi.upper_left_latitude > 82 if HEMISPHERE == 'north' else requested_aoi.lower_right_latitude < -82):
    raise ValueError("AOI must reach the selected polar hemisphere; crossing its 82° seam is allowed.")
static_only = MODALITY == "static"
source_grid = None
selectors = ()
GROUP_NAME = "polar"
if not static_only:
    if SOURCE_RASTER is None:
        if MODALITY != "wac" or WAC_LAYOUT != "single":
            raise ValueError("Set SOURCE_RASTER to your original imagery raster.")
        matches = sorted(POLAR_WAC_DIR.glob(f"*{expected_family.upper()}.vrt"))
        if len(matches) != 1:
            raise ValueError(f"Expected one polar VRT; found {len(matches)}. Set SOURCE_RASTER explicitly.")
        SOURCE_RASTER = matches[0]
    SOURCE_RASTER = Path(SOURCE_RASTER)
    source_grid = read_source_grid(
        SOURCE_RASTER, expected_band_count=5 if MODALITY == "wac" and WAC_LAYOUT == "vis_uv" else 1)
    PRODUCT_ID = lunar_product_id_from_raster_path(SOURCE_RASTER)
    selectors = (SourceSelector(GROUP_NAME, MODALITY, PRODUCT_ID),)
LABEL_PATH = latest_crater_label_path() if LABEL_PATH is None else Path(LABEL_PATH)
if not LABEL_PATH.is_file():
    raise FileNotFoundError(LABEL_PATH)
label_input = LabelInput(LABEL_PATH, kind=LABEL_KIND, relation="clip_to_target",
                         layer=LABEL_LAYER if LABEL_PATH.suffix.lower() == ".gpkg" else None)
request = chip_request_from_aoi(
    sample_id=SAMPLE_ID, geographic_aoi=requested_aoi,
    source_grid=source_grid, static_only=static_only,
    split_group_key=SAMPLE_ID if static_only else PRODUCT_ID,
    source_selectors=selectors, label_input=label_input,
)
if request.target_grid.width * request.target_grid.height > MAX_OUTPUT_PIXELS:
    raise ValueError("Output exceeds MAX_OUTPUT_PIXELS; reduce AOI or explicitly raise the limit.")
ZOOM_LEVEL = default_chip_zoom(request.geographic_aoi, MODALITY) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
LTM_ZOOM = (11 if MODALITY == 'nac' else 5) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
POLAR_ZOOM = (10 if MODALITY == 'nac' else 4) if ZOOM_OVERRIDE is None else ZOOM_OVERRIDE
OUTPUT_ROOT = OUTPUT_BASE_DIR / datetime.now().strftime("%Y%m%dT%H%M%S_%f")
print(f"Labels: {LABEL_PATH}")
print(f"Center/sole family: {family.value} | acquisition zooms: LTM={LTM_ZOOM}, polar={POLAR_ZOOM}")
print(f"Output: {request.target_grid.width} x {request.target_grid.height} | affine: {request.target_grid.transform}")
print(f"Destination: {OUTPUT_ROOT}")
requests = [request]


### Additional query: labeled north seam

This reuses the source grid, labels and product selector, with a distinct sample ID. The default AOI includes craters 13–20 from `(2).gpkg` and extends below 82°N (about 2.2 million pixels at 100 m). No supplied outlines cross the seam itself: inspect imagery continuity there and label alignment farther north. The output rectangle may extend beyond the geographic AOI. Both results share one manifest and are plotted below.


In [ ]:
requests = [request]  # Reset so rerunning this cell never duplicates the query.
if RUN_SEAM_EXAMPLE:
    seam_aoi = GeographicAOI(*SEAM_AOI_NWSE)
    boundary = 82 if HEMISPHERE == "north" else -82
    if not seam_aoi.lower_right_latitude < boundary < seam_aoi.upper_left_latitude:
        raise ValueError("SEAM_AOI_NWSE must cross the selected hemisphere's 82° seam.")
    seam_request = chip_request_from_aoi(
        sample_id=f"{SAMPLE_ID}_seam", geographic_aoi=seam_aoi,
        source_grid=source_grid, static_only=static_only,
        split_group_key=request.split_group_key,
        source_selectors=selectors, label_input=label_input,
    )
    grid = seam_request.target_grid
    if grid.width * grid.height > MAX_OUTPUT_PIXELS:
        raise ValueError("Seam output exceeds MAX_OUTPUT_PIXELS; reduce SEAM_AOI_NWSE.")
    requests.append(seam_request)
    print(f"Seam query: {SEAM_AOI_NWSE} | {grid.width} x {grid.height} pixels")


### Optional query: pole-containing rectangle

Enable `RUN_POLE_EXAMPLE` only with imagery and finished labels covering the pole. A full-longitude cap becomes an enclosing native-grid rectangle: corner imagery and labels outside the cap are retained, not masked. Acquisition covers that entire rectangle. The supplied example annotations are farther from the pole, so an all-background result here is not evidence of a correctly annotated negative sample. Large rectangles extending below the polar coverage boundary are rejected.


In [ ]:
requests = [item for item in requests if item.sample_id != f"{SAMPLE_ID}_pole"]
if RUN_POLE_EXAMPLE:
    pole_aoi = GeographicAOI(*POLE_AOI_NWSE)
    pole_latitude = pole_aoi.upper_left_latitude if HEMISPHERE == "north" else pole_aoi.lower_right_latitude
    if pole_latitude != (90 if HEMISPHERE == "north" else -90):
        raise ValueError("POLE_AOI_NWSE must reach the selected hemisphere's pole.")
    pole_request = chip_request_from_aoi(
        sample_id=f"{SAMPLE_ID}_pole", geographic_aoi=pole_aoi,
        source_grid=source_grid, static_only=static_only,
        split_group_key=request.split_group_key,
        source_selectors=selectors, label_input=label_input,
    )
    grid = pole_request.target_grid
    if grid.width * grid.height > MAX_OUTPUT_PIXELS:
        raise ValueError("Pole output exceeds MAX_OUTPUT_PIXELS; reduce the cap.")
    requests.append(pole_request)
    print(f"Pole query: {POLE_AOI_NWSE} | {grid.width} x {grid.height} pixels; corners retained")
    print("Acquisition envelope:", pole_request.geographic_aoi)


## 3. Prepare enabled indexes and configure chips

Dynamic indexing is restricted to the chosen file/product and stored in this run's directory. Shared static indexes remain read-only; custom static collections use a per-clone cache. Disabled collections are not inspected. Single-band polar WAC is explicitly selected as one band, never forced into the seven-band contract.


In [ ]:
sources, modalities = [], []
if not static_only:
    pattern = DYNAMIC_GLOB or (
        f"{PRODUCT_ID}.prj.*.mos.tif" if MODALITY == "wac" and WAC_LAYOUT == "vis_uv"
        else SOURCE_RASTER.name)
    index_config = VectorIndexBuildConfig(
        data_dir=SOURCE_RASTER.parent, index_path=OUTPUT_ROOT / "indexes/dynamic.gpkg",
        image_glob=pattern, worker_count=INDEX_WORKER_COUNT)
    if SOURCE_RASTER.resolve() not in {p.resolve() for p in discover_raster_paths(index_config)}:
        raise ValueError('DYNAMIC_GLOB must include SOURCE_RASTER.')
    index = ensure_vector_index(index_config, stdout=sys.stdout).index_path
    sources.append(TileSourceConfig(
        name=MODALITY, data_dir=SOURCE_RASTER.parent, index_path=index,
        selection_mode="product_id", preserve_source_nodata=True, required=True))
    single = MODALITY == "nac" or WAC_LAYOUT == "single"
    modalities.append(OutputModalityConfig(
        GROUP_NAME, MODALITY, MODALITY,
        band_indices=(1,) if single else None,
        output_band_names=(MODALITY,) if single else None))
if static_only or INCLUDE_STATIC:
    resolution = resolve_notebook_source_index(
        source_name="static", data_dir=STATIC_DATA_DIR, cache_dir=OUTPUT_BASE_DIR / "indexes")
    index = ensure_vector_index(VectorIndexBuildConfig(
        data_dir=resolution.data_dir, index_path=resolution.index_path,
        rebuild_invalid_index=resolution.rebuild_invalid_index,
        worker_count=INDEX_WORKER_COUNT), stdout=sys.stdout).index_path
    sources.append(make_static_source(data_dir=STATIC_DATA_DIR, index_path=index))
    modalities.append(OutputModalityConfig(GROUP_NAME, "static", "static"))
chip_config = ChipConfig(
    output_root=OUTPUT_ROOT, label_source=LABEL_PATH,
    acquisition_groups=(AcquisitionGroupConfig(
        GROUP_NAME, TileConfig(OUTPUT_ROOT / ".tiling-template", ZOOM_LEVEL, tuple(sources)),
        ltm_zoom_level=LTM_ZOOM, polar_zoom_level=POLAR_ZOOM),),
    output_modalities=tuple(modalities), split_config=SPLIT_CONFIG,
)


## 4. Create and inspect

Per-sample errors are reported without silently treating failures as successful chips. Review the manifest, warnings and overlay. Run north/south, dynamic-only/mixed/static-only cases separately before relying on a new collection.


In [ ]:
batch = create_chips(requests, chip_config, max_workers=MAX_WORKERS)
print(f"Manifest: {batch.manifest_path}")
for result in batch.results:
    print(result.request.sample_id, result.status, result.message)
    for diagnostic in result.diagnostics:
        print(f"{diagnostic.stage}/{diagnostic.code}: {diagnostic.message}")
    if result.status == "success":
        plot_chip_result(result, display_band_keyword="elv" if static_only else MODALITY)
